In [ ]:
# Setup -- SQL right inside this notebook via sqlite3
import sqlite3
import pandas as pd
import os

conn = sqlite3.connect(':memory:')

def q(sql):
    return pd.read_sql_query(sql, conn)

def run(sql):
    conn.executescript(sql)
    conn.commit()

# Path to the 4 CSVs (downloaded from the course site)
DATA_DIR = os.path.expanduser('./data')

print('SQL engine ready.  sqlite3 version:', sqlite3.sqlite_version)
print('Data dir            :', DATA_DIR)
print('Files present       :', sorted(os.listdir(DATA_DIR)) if os.path.isdir(DATA_DIR) else 'NOT FOUND')

SQL engine ready.  sqlite3 version: 3.37.2
Data dir            : ./data
Files present       : ['analysis.sql', 'companies.csv', 'data_cleaning.sql', 'employees.CSV', 'functions.csv', 'salaries.csv']


In [ ]:
# Load the four CSVs into the SQLite engine
import pandas as pd

# Note: encoding='utf-8-sig' handles the byte-order-mark (BOM) on companies.csv
# Note: decimal=',' converts European-style "6335,56" -> 6335.56 on the salary column
companies = pd.read_csv(f'{DATA_DIR}/companies.csv',
                        sep=';', encoding='utf-8-sig')
employees = pd.read_csv(f'{DATA_DIR}/employees.CSV',
                        sep=';', encoding='utf-8-sig')
functions = pd.read_csv(f'{DATA_DIR}/functions.csv',
                        sep=';', encoding='utf-8-sig')
salaries  = pd.read_csv(f'{DATA_DIR}/salaries.csv',
                        sep=';', encoding='utf-8-sig', decimal=',')

# Drop any pre-existing tables (re-run safety)
run('DROP TABLE IF EXISTS companies; DROP TABLE IF EXISTS employees; '
    'DROP TABLE IF EXISTS functions; DROP TABLE IF EXISTS salaries; '
    'DROP TABLE IF EXISTS df_employee;')

# Push DataFrames into SQLite
companies.to_sql('companies', conn, index=False)
employees.to_sql('employees', conn, index=False)
functions.to_sql('functions', conn, index=False)
salaries.to_sql('salaries',   conn, index=False)

# Sanity-check row counts
print('companies :', len(companies))
print('employees :', len(employees))
print('functions :', len(functions))
print('salaries  :', len(salaries))

companies : 25
employees : 1156
functions : 107
salaries  : 8049


In [ ]:
q('''

SELECT *
FROM salaries
group by comp_code,comp_name,employee_id,employee_name,date,func_code,func,salary
having count(*)>1
''')

,comp_code,comp_name,employee_id,employee_name,date,func_code,func,salary
0,1,The Crossings at Falcon Point,18009,Neil Mejia,01/09/2022 00:00,2,Bricklayer,1870.0
1,1,The Crossings at Falcon Point,18044,Jarrod Bauer,01/09/2022 00:00,2,Bricklayer,1870.0
2,1,The Crossings at Falcon Point,18144,Ezequiel McKenzie,01/09/2022 00:00,10,Watchman,1221.0
3,1,The Crossings at Falcon Point,23329,Brayan Saunders,01/09/2022 00:00,101,Tower Crane Operator,1870.0
4,1,The Crossings at Falcon Point,23339,Allan Cannon,01/09/2022 00:00,97,Construction Foreman I,2013.2
...,...,...,...,...,...,...,...,...
181,83,DM Company Head Quarters,28590,William Wilson,01/01/2022 00:00,64,Accounting Assistant I,2500.0
182,83,DM Company Head Quarters,29060,Justin Martin,01/01/2022 00:00,84,Purchaser,4300.0
183,83,DM Company Head Quarters,29648,Zachary Taylor,01/01/2022 00:00,43,Payroll Department Assistant,1600.0
184,83,DM Company Head Quarters,29765,Brandon Martinez,01/01/2022 00:00,111,Administrative Assistant,1500.0


In [ ]:
run('''
DELETE FROM salaries
WHERE rowid NOT IN (
    SELECT MIN(rowid)
    FROM salaries
    GROUP BY
        comp_code,
        comp_name,
        employee_id,
        employee_name,
        date,
        func_code,
        func,
        salary
);
''')

In [ ]:
q('''

SELECT *
FROM salaries
group by comp_code,comp_name,employee_id,employee_name,date,func_code,func,salary
having count(*)>1
''')

,comp_code,comp_name,employee_id,employee_name,date,func_code,func,salary


In [ ]:
# Exercise 1: Building a Comprehensive Dataset for Employee Analysis
# Create a temporary table that join all tables and create a new one using LEFT JOIN.
# Create an unique identifier code between the columns ‘employee_id’ and ‘date’ and call it ‘id’.
# Convert the column ‘date’ to DATE type because it was previously configured as TIMESTAMP.
# Transform this new table into a dataset “df_employee” for analysis.


run('''
CREATE TABLE df_employee AS
SELECT
    s.employee_id || '_' || s.date AS id,
    DATE(SUBSTR(s.date, 7, 4) || '-' || SUBSTR(s.date, 4, 2) || '-' || SUBSTR(s.date, 1, 2)) AS month_year,
    s.employee_id,
    s.employee_name,
    e."GEN(M_F)"      AS gender,
    e.age,
    s.salary,
    f.function_group,
    c.company_name,
    c.company_city,
    c.company_state,
    c.company_type,
    c.const_site_category
FROM salaries s
LEFT JOIN employees e ON e.employee_code_emp = s.employee_id
LEFT JOIN functions f ON f.function_code     = s.func_code
LEFT JOIN companies c ON c.company_name      = s.comp_name
''')

q('SELECT * FROM df_employee LIMIT 5')

,id,month_year,employee_id,employee_name,gender,age,salary,function_group,company_name,company_city,company_state,company_type,const_site_category
0,26193_01/01/2022 00:00,2022-01-01,26193,Jacob Smith,M,38,6335.56,Managers,DM Company Head Quarters,Goiania,GOIAS,Administration,None
1,25322_01/01/2022 00:00,2022-01-01,25322,Michael Johnson,M,30,2619.35,Administration,DM Company Head Quarters,Goiania,GOIAS,Administration,None
2,27602_01/01/2022 00:00,2022-01-01,27602,Matthew Williams,M,28,1221.67,Administration,DM Company Head Quarters,Goiania,GOIAS,Administration,None
3,27127_01/01/2022 00:00,2022-01-01,27127,Joshua Brown,M,25,4000.00,Engineering,DM Company Head Quarters,Goiania,GOIAS,Administration,None
4,23007_01/01/2022 00:00,2022-01-01,23007,Christopher Jones,M,25,3012.52,Administration,DM Company Head Quarters,Goiania,GOIAS,Administration,None


In [ ]:
# Exercise 2: Cleaning Data for Consistency and Quality
# 1. run the following SQLite request and observe your new table.

# SELECT * FROM df_employee;

q('''SELECT * FROM df_employee LIMIT 20;''')


# 2. Remove all unwanted spaces from all text columns using TRIM

run('''
UPDATE df_employee
SET id            = TRIM(id),
    employee_name = TRIM(employee_name),
    function_group = TRIM(function_group),
    company_name   = TRIM(company_name),
    company_city   = TRIM(company_city),
    company_state  = TRIM(company_state),
    company_type   = TRIM(company_type)
''')


# 3. Check for NULL values and empty values.

q('''
SELECT *
FROM df_employee
WHERE id IS NULL
   OR LOWER(TRIM(id)) IN ('null', 'unknown', 'n/a', '')

   OR employee_name IS NULL
   OR LOWER(TRIM(employee_name)) IN ('null', 'unknown', 'n/a', '')

   OR salary IS NULL

   OR gender IS NULL
   OR LOWER(TRIM(gender)) IN ('null', 'unknown', 'n/a', '')

   OR age IS NULL

   OR function_group IS NULL
   OR LOWER(TRIM(function_group)) IN ('null', 'unknown', 'n/a', '')

   OR company_name IS NULL
   OR LOWER(TRIM(company_name)) IN ('null', 'unknown', 'n/a', '')

   OR company_city IS NULL
   OR LOWER(TRIM(company_city)) IN ('null', 'unknown', 'n/a', '')

   OR company_state IS NULL
   OR LOWER(TRIM(company_state)) IN ('null', 'unknown', 'n/a', '')

   OR company_type IS NULL
   OR LOWER(TRIM(company_type)) IN ('null', 'unknown', 'n/a', '')

   OR const_site_category IS NULL
   OR LOWER(TRIM(const_site_category)) IN ('null', 'unknown', 'n/a', '');
''')


# 4. Delete rows of the detected missing values.

run('''DELETE FROM df_employee
WHERE id IS NULL
   OR LOWER(TRIM(id)) IN ('null', 'unknown', 'n/a', '')

   OR employee_name IS NULL
   OR LOWER(TRIM(employee_name)) IN ('null', 'unknown', 'n/a', '')

   OR salary IS NULL

   OR gender IS NULL
   OR LOWER(TRIM(gender)) IN ('null', 'unknown', 'n/a', '')

   OR age IS NULL

   OR function_group IS NULL
   OR LOWER(TRIM(function_group)) IN ('null', 'unknown', 'n/a', '')

   OR company_name IS NULL
   OR LOWER(TRIM(company_name)) IN ('null', 'unknown', 'n/a', '')

   OR company_city IS NULL
   OR LOWER(TRIM(company_city)) IN ('null', 'unknown', 'n/a', '')

   OR company_state IS NULL
   OR LOWER(TRIM(company_state)) IN ('null', 'unknown', 'n/a', '')

   OR company_type IS NULL
   OR LOWER(TRIM(company_type)) IN ('null', 'unknown', 'n/a', '')

   OR const_site_category IS NULL
   OR LOWER(TRIM(const_site_category)) IN ('null', 'unknown', 'n/a', '');
''')





In [ ]:
q('''
SELECT *
from df_employee

''')

,id,month_year,employee_id,employee_name,gender,age,salary,function_group,company_name,company_city,company_state,company_type,const_site_category
0,18015_01/01/2022 00:00,2022-01-01,18015,Brendan Foster,M,21,1212.00,Assistants,The Grove at Parkside,Palmas,Tocantins,Construction Site,Residential
1,22983_01/01/2022 00:00,2022-01-01,22983,Jeremiah Sanders,M,25,1212.00,Assistants,The Grove at Parkside,Palmas,Tocantins,Construction Site,Residential
2,18085_01/01/2022 00:00,2022-01-01,18085,Xavier Ross,M,24,1212.00,Assistants,The Grove at Parkside,Palmas,Tocantins,Construction Site,Residential
3,17942_01/01/2022 00:00,2022-01-01,17942,Jorge Russell,M,34,1739.30,Professionals,The Summit at Laurel Canyon,Palmas,Tocantins,Construction Site,Commercial
4,22422_01/01/2022 00:00,2022-01-01,22422,Edward Ortiz,M,18,1212.00,Assistants,The Summit at Laurel Canyon,Palmas,Tocantins,Construction Site,Commercial
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6808,33082_01/01/2023 00:00,2023-01-01,33082,Wendy Matthews,F,18,1286.29,Assistants,The Sanctuary at Briarcliff,Brasilia,Distrito Federal,Construction Site,Residential
6809,32986_01/01/2023 00:00,2023-01-01,32986,Fatima Mason,F,20,1375.00,Assistants,The Sanctuary at Briarcliff,Brasilia,Distrito Federal,Construction Site,Residential
6810,23382_01/01/2023 00:00,2023-01-01,23382,Marques Sampson,M,28,1254.73,Assistants,The Sanctuary at Briarcliff,Brasilia,Distrito Federal,Construction Site,Residential
6811,31731_01/01/2023 00:00,2023-01-01,31731,Andreas Oconnell,M,21,1000000.00,Assistants,The Sanctuary at Briarcliff,Brasilia,Distrito Federal,Construction Site,Residential


In [ ]:
# Exercise 3 : Calculating Current Employee Counts by Company
# How many employees do the companies have today?
# Group them by company

q('''
SELECT MAX(month_year) AS latest_month
FROM df_employee;
''')

q('''
SELECT
    company_name,
    COUNT(DISTINCT employee_id) AS current_employees
FROM df_employee
WHERE month_year = (
    SELECT MAX(month_year)
    FROM df_employee
)
GROUP BY company_name
ORDER BY current_employees DESC;
''')

,company_name,current_employees
0,The Crossings at Falcon Point,107
1,Regional Hospital,82
2,The Terraces at Diamond Heights,75
3,The Glades at Maplewood,60
4,The Pines at Windward,57
5,The Greens at Fairway Hills,57
6,The Sanctuary at Briarcliff,43
7,The Oasis at Desert Springs,39
8,The Meadows at Sunset Ridge,39
9,The Landings at Ocean Breeze,4


In [ ]:
# Exercise 4 : Analyzing Employee Distribution by City and Over Time
# What is the total number of employees each city? Add a percentage column


q('''
SELECT
    company_city,
    COUNT(DISTINCT employee_id) AS total_employees,
    ROUND(
        COUNT(DISTINCT employee_id) * 100.0
        / SUM(COUNT(DISTINCT employee_id)) OVER (),
        2
    ) AS percentage
FROM df_employee
GROUP BY company_city
ORDER BY total_employees DESC;
''')




,company_city,total_employees,percentage
0,Goiania,622,59.52
1,Brasilia,371,35.50
2,Palmas,50,4.78
3,Goianiaa,2,0.19


In [ ]:
# What is the total number of employees each month?

q('''
SELECT
    month_year,
    COUNT(DISTINCT employee_id) AS total_employees
FROM df_employee
GROUP BY month_year
ORDER BY month_year;
''')

,month_year,total_employees
0,2022-01-01,438
1,2022-02-01,447
2,2022-03-01,451
3,2022-04-01,527
4,2022-05-01,533
5,2022-06-01,550
6,2022-07-01,525
7,2022-08-01,539
8,2022-09-01,555
9,2022-10-01,513


In [ ]:
# What is the average number of employees each month?

q('''
SELECT
    ROUND(AVG(total_employees), 2) AS average_employees_per_month
FROM (
    SELECT
        month_year,
        COUNT(DISTINCT employee_id) AS total_employees
    FROM df_employee
    GROUP BY month_year
) monthly_counts;
''')

,average_employees_per_month
0,524.08


In [ ]:
# Minimum and maximum number of employees throughout all months
q('''
WITH monthly_counts AS (
    SELECT
        month_year,
        COUNT(DISTINCT employee_id) AS total_employees
    FROM df_employee
    GROUP BY month_year
),
min_max_values AS (
    SELECT
        MIN(total_employees) AS min_employees,
        MAX(total_employees) AS max_employees
    FROM monthly_counts
)
SELECT
    mc.month_year,
    mc.total_employees,
    CASE
        WHEN mc.total_employees = mmv.min_employees THEN 'Minimum'
        WHEN mc.total_employees = mmv.max_employees THEN 'Maximum'
    END AS type
FROM monthly_counts mc
CROSS JOIN min_max_values mmv
WHERE mc.total_employees = mmv.min_employees
   OR mc.total_employees = mmv.max_employees
ORDER BY mc.total_employees;
''')


,month_year,total_employees,type
0,2022-01-01,438,Minimum
1,2022-11-01,581,Maximum


In [ ]:
# Monthly average number of employees by function group

q('''
WITH monthly_function_counts AS (
    SELECT
        month_year,
        function_group,
        COUNT(DISTINCT employee_id) AS total_employees
    FROM df_employee
    GROUP BY month_year, function_group
)
SELECT
    function_group,
    ROUND(AVG(total_employees), 2) AS avg_employees_per_month
FROM monthly_function_counts
GROUP BY function_group
ORDER BY avg_employees_per_month DESC;
''')

,function_group,avg_employees_per_month
0,Professionals,244.62
1,Assistants,200.62
2,Production Supervisors,28.54
3,Administration,16.15
4,Machine Operators,15.92
5,Engineering,9.46
6,Trainees,8.77


In [ ]:
# Annual average salary
q('''
SELECT
    STRFTIME('%Y', month_year) AS year,
    ROUND(AVG(salary), 2) AS annual_average_salary
FROM df_employee
GROUP BY STRFTIME('%Y', month_year)
ORDER BY year;
''')

,year,annual_average_salary
0,2022,1603.60
1,2023,11963.45
